# 03c — Case-Control Matching (Problem C)

Match NTSB incidents (cases) to BTS normal flights (controls).  
This creates the labeled dataset for training the pre-flight risk model.

**Design:** 1:20 case-control ratio  
**Output:** `data/processed/preflight_casecontrol.parquet`

In [1]:
import os
from pathlib import Path

root = Path.cwd()
while not (root / 'pyproject.toml').exists():
    root = root.parent
os.chdir(root)
print(f'Working directory: {root}')

Working directory: e:\OSFDA


In [2]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

NTSB_PATH = Path('data/raw/ntsb_accidents.parquet')
BTS_DIR   = Path('data/raw/bts_flights')
OUT_DIR   = Path('data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONTROL_RATIO = 20  # controls per case
SEED = 42

print(f'Case-control ratio: 1:{CONTROL_RATIO}')

Case-control ratio: 1:20


## 1. Load NTSB Cases

In [3]:
ntsb = pd.read_parquet(NTSB_PATH)
print(f'NTSB events: {len(ntsb):,}')

# Ensure event_date is datetime
ntsb['event_date'] = pd.to_datetime(ntsb['event_date'], errors='coerce')

# ---------- Airport Code Resolution ----------
# NTSB schema has two useful airport fields:
#   dprt_apt_id  : FAA identifier of DEPARTURE airport (best for commercial flight matching)
#   ev_nr_apt_id : Nearest airport to the event site (often general aviation)
# BTS uses 3-letter IATA codes (ATL, ORD, LAX…).
# Strategy: prefer dprt_apt_id, fall back to ev_nr_apt_id.
# Normalization: strip whitespace, uppercase, remove leading 'K' from 4-char ICAO codes.

def _normalize_apt(s):
    """Normalize FAA/ICAO airport code to 3-letter IATA equivalent."""
    if pd.isna(s):
        return None
    s = re.sub(r'\s+', '', str(s)).upper()  # strip all whitespace
    s = re.sub(r'^K([A-Z]{3})$', r'\1', s)  # KORD -> ORD
    return s if s else None

ntsb['dprt_code'] = ntsb['dprt_apt_id'].apply(_normalize_apt)
ntsb['ev_code']   = ntsb['ev_nr_apt_id'].apply(_normalize_apt)

# We'll resolve the best code after loading BTS (to know which codes are valid)
print('NTSB loaded. Airport code columns created.')
print(f'  dprt_apt_id coverage: {ntsb["dprt_code"].notna().sum():,}')
print(f'  ev_nr_apt_id coverage: {ntsb["ev_code"].notna().sum():,}')

NTSB events: 4,704
NTSB loaded. Airport code columns created.
  dprt_apt_id coverage: 3,498
  ev_nr_apt_id coverage: 2,821


## 2. Load BTS Flights

In [4]:
# Load BTS yearly parquets with downcasting to save memory
bts_files = sorted(BTS_DIR.glob('bts_20??.parquet'))
print(f'BTS files found: {[f.name for f in bts_files]}')

bts_frames = []
for f in bts_files:
    df = pd.read_parquet(f)

    # Downcast numeric columns to save space
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')

    # Keep ORIGIN as plain object string (not category) for reliable merging
    for col in ['ORIGIN_STATE_ABR', 'DEST_STATE_ABR']:
        if col in df.columns:
            df[col] = df[col].astype('category')

    print(f'  {f.name}: {len(df):,} flights')
    bts_frames.append(df)

bts = pd.concat(bts_frames, ignore_index=True)
bts['FL_DATE'] = pd.to_datetime(bts['FL_DATE'], errors='coerce')
print(f'\nTotal BTS flights: {len(bts):,}')

# Build the set of valid BTS airport codes for matching
bts_origins_set = set(bts['ORIGIN'].dropna().unique())
print(f'Unique BTS origin airports: {len(bts_origins_set)}')

BTS files found: ['bts_2018.parquet', 'bts_2019.parquet', 'bts_2020.parquet']
  bts_2018.parquet: 7,206,195 flights
  bts_2019.parquet: 7,422,037 flights
  bts_2020.parquet: 4,688,354 flights

Total BTS flights: 19,316,586
Unique BTS origin airports: 373


## 3. Resolve NTSB Airport Codes & Match to BTS

We match on **date + origin airport**. A match means an NTSB incident
occurred at the same airport on the same day as a scheduled BTS flight.

We prefer `dprt_apt_id` (departure airport) over `ev_nr_apt_id` (nearest airport)
because BTS only records commercial departure airports.

In [5]:
# Resolve airport_code: prefer departure airport if it's in BTS, else event airport
ntsb['airport_code'] = np.where(
    ntsb['dprt_code'].isin(bts_origins_set),
    ntsb['dprt_code'],
    ntsb['ev_code']
)

# Filter to events that have a BTS-matchable airport
ntsb_matched = ntsb[ntsb['airport_code'].isin(bts_origins_set)].copy()
print(f'Matchable NTSB events: {len(ntsb_matched):,} / {len(ntsb):,}')

# Create matching keys (date + airport)
ntsb_matched['match_date'] = ntsb_matched['event_date'].dt.date
bts['match_date'] = bts['FL_DATE'].dt.date

# Create a unique (date, airport) incident-day lookup
incident_keys_df = (
    ntsb_matched[['match_date', 'airport_code']]
    .dropna()
    .drop_duplicates()
    .assign(is_incident_day=True)
)
print(f'Unique incident (date, airport) pairs: {len(incident_keys_df):,}')

# Merge with BTS to flag incident-day flights
# ORIGIN is object dtype — no category casting issue
bts = bts.merge(
    incident_keys_df,
    left_on=['match_date', 'ORIGIN'],
    right_on=['match_date', 'airport_code'],
    how='left'
)
# Fix FutureWarning: explicit bool cast instead of relying on fillna inference
bts['is_incident_day'] = bts['is_incident_day'].fillna(False).astype(bool)

# These are the 'cases' — commercial flights on incident days/airports
incident_flights = bts[bts['is_incident_day']].copy()
print(f'BTS flights on incident days: {len(incident_flights):,}')

Matchable NTSB events: 607 / 4,704
Unique incident (date, airport) pairs: 606


C:\Users\yashn\AppData\Local\Temp\ipykernel_17968\3885992723.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bts['is_incident_day'] = bts['is_incident_day'].fillna(False).astype(bool)


BTS flights on incident days: 48,202


## 4. Construct Case-Control Dataset

In [6]:
# 1. Prepare Cases (Positive samples)
cases = incident_flights.copy()
cases['incident'] = 1
n_cases = len(cases)
print(f'Cases: {n_cases:,}')

# 2. Setup Sampling parameters
n_controls_needed = n_cases * CONTROL_RATIO
print(f'Controls needed: {n_controls_needed:,} (1:{CONTROL_RATIO} ratio)')

# 3. Add temporal markers for stratified sampling
bts['year_month'] = bts['FL_DATE'].dt.to_period('M')
cases['year_month'] = cases['FL_DATE'].dt.to_period('M')

# Control pool: BTS flights NOT on incident days
control_pool = bts[~bts['is_incident_day']].copy()
control_pool['year_month'] = control_pool['FL_DATE'].dt.to_period('M')

# 4. Perform Stratified Sampling (match monthly distribution of cases)
case_month_dist = cases['year_month'].value_counts(normalize=True)
control_samples = []

rng = np.random.RandomState(SEED)
for ym, frac in case_month_dist.items():
    n_sample = int(n_controls_needed * frac) + 1
    pool_subset = control_pool[control_pool['year_month'] == ym]
    n_actual = min(n_sample, len(pool_subset))
    if n_actual > 0:
        sampled = pool_subset.sample(n=n_actual, random_state=rng)
        control_samples.append(sampled)

# 5. Finalize Controls
controls = pd.concat(control_samples, ignore_index=True)
controls['incident'] = 0
print(f'Controls sampled: {len(controls):,}')

# 6. Combine and Cleanup
dataset = pd.concat([cases, controls], ignore_index=True)
drop_cols = ['is_incident_day', 'match_date', 'year_month', 'dprt_code', 'ev_code']
dataset = dataset.drop(columns=[c for c in drop_cols if c in dataset.columns])

# Shuffle final dataset
dataset = dataset.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'\nFinal dataset: {len(dataset):,} rows')
print(f'Incident rate: {dataset["incident"].mean()*100:.2f}%')

Cases: 48,202
Controls needed: 964,040 (1:20 ratio)
Controls sampled: 964,076

Final dataset: 1,012,278 rows
Incident rate: 4.76%


## 5. Validate

In [7]:
print('Label distribution:')
print(dataset['incident'].value_counts())

print(f'\nTemporal range: {dataset["FL_DATE"].min()} to {dataset["FL_DATE"].max()}')
print(f'Unique origins: {dataset["ORIGIN"].nunique()}')
print(f'Unique carriers: {dataset["OP_UNIQUE_CARRIER"].nunique()}')

# Sanity: cases should span the full date range
case_dates = dataset[dataset['incident'] == 1]['FL_DATE']
print(f'\nCase date range: {case_dates.min()} to {case_dates.max()}')

# Year distribution
print('\nRows per year:')
print(dataset['FL_DATE'].dt.year.value_counts().sort_index())

Label distribution:
incident
0    964076
1     48202
Name: count, dtype: int64

Temporal range: 2018-01-01 00:00:00 to 2020-12-31 00:00:00
Unique origins: 371
Unique carriers: 18

Case date range: 2018-01-01 00:00:00 to 2020-12-30 00:00:00

Rows per year:
FL_DATE
2018    458862
2019    400398
2020    153018
Name: count, dtype: int64


In [8]:
# Save
out_path = OUT_DIR / 'preflight_casecontrol.parquet'
dataset.to_parquet(out_path, index=False)
print(f'Saved {len(dataset):,} rows to {out_path}')
print(f'File size: {out_path.stat().st_size / 1e6:.1f} MB')

Saved 1,012,278 rows to data\processed\preflight_casecontrol.parquet
File size: 18.6 MB
